In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import imageio

#############################
# Global random map
#############################
np.random.seed(0)
global_random_map = np.random.rand(512, 512)  # Pre-generate up to 512×512

#############################
# Helper: fill region with rough terrain
#############################
def fill_with_rough_terrain(heightmap, x_start, x_end, y_start, y_end, roughness):
    """
    Fill [y_start:y_end, x_start:x_end] with random noise scaled by 'roughness' in [0..1].
    """
    region_h = y_end - y_start
    region_w = x_end - x_start
    rough_chunk = global_random_map[:region_h, :region_w]
    heightmap[y_start:y_end, x_start:x_end] = rough_chunk * roughness

#############################
# Helper: place one symmetric "up-down" stair rectangle
#############################
def fill_one_stair_rectangle(heightmap, occupancy,
                             x_start, x_end, 
                             y_start, y_end,
                             rect_x, rect_y, rect_w, rect_h,
                             stair_height):
    """
    Place one rectangular region (rect_w x rect_h) containing a *symmetric* up/down ramp.
    - We pick n_up in [2..5]; then n_down = n_up.
    - The ramp does NOT start at fraction=0 (so no wide flat region).
    - Orientation is chosen randomly: "horizontal" or "vertical."

    If ANY pixel is already occupied, we skip (return False).
    """

    # Bounds check
    rect_x2 = min(rect_x + rect_w, x_end)
    rect_y2 = min(rect_y + rect_h, y_end)
    if rect_x2 <= rect_x or rect_y2 <= rect_y:
        return False  # out of bounds or zero area

    # Convert to local occupancy indexing
    occ_x0 = rect_x - x_start
    occ_x1 = rect_x2 - x_start
    occ_y0 = rect_y - y_start
    occ_y1 = rect_y2 - y_start
    region_occupied = occupancy[occ_y0:occ_y1, occ_x0:occ_x1]
    if np.any(region_occupied):
        return False  # overlap => skip

    # Choose random orientation
    orientation = np.random.choice(["horizontal", "vertical"])

    # Number of steps up == number of steps down
    n_up = np.random.randint(3, 7)  # 2..6
    n_down = n_up
    # Build an array of fractions (not starting at 0) that go up, then go down:
    # Up steps: [1/n_up, 2/n_up, ..., n_up/n_up=1]
    fractions_up = [(i+1)/float(n_up) for i in range(n_up)]
    # Down steps: [1 - 1/n_down, 1 - 2/n_down, ..., 1 - n_down/n_down=0]
    fractions_down = [1.0 - i/float(n_down) for i in range(1, n_down)]
    
    fractions = fractions_up + fractions_down  # total steps = 2*n_up in [2..10]

    if orientation == "horizontal":
        total_w = rect_x2 - rect_x
        step_w = total_w / float(len(fractions))
        for step_idx, frac in enumerate(fractions):
            hval = frac * stair_height
            x0 = int(rect_x + step_idx * step_w)
            x1 = int(rect_x + (step_idx+1) * step_w)
            y0 = rect_y
            y1 = rect_y2
            heightmap[y0:y1, x0:x1] = hval

    else:
        # orientation == "vertical"
        total_h = rect_y2 - rect_y
        step_h = total_h / float(len(fractions))
        for step_idx, frac in enumerate(fractions):
            hval = frac * stair_height
            x0 = rect_x
            x1 = rect_x2
            y0 = int(rect_y + step_idx * step_h)
            y1 = int(rect_y + (step_idx+1) * step_h)
            heightmap[y0:y1, x0:x1] = hval

    # Mark occupancy
    occupancy[occ_y0:occ_y1, occ_x0:occ_x1] = True
    return True

#############################
# Main fill_staircase_region
#############################
def fill_staircase_region(heightmap,
                          x_start, x_end, 
                          y_start, y_end,
                          coverage,     # fraction of this region to fill with stairs
                          stair_height, # total height [0..1]
                          stair_length, # horizontal size of each rectangle
                          stair_width,  # vertical size of each rectangle
                          leftover_roughness):
    """
    1) Fill entire region with leftover rough terrain (scaled by leftover_roughness).
    2) Then randomly place symmetric "up-down" rectangles until coverage fraction or max attempts.
    3) margin=10 around edges to avoid partial ramps out-of-bounds in all directions.
    """
    fill_with_rough_terrain(heightmap, x_start, x_end, y_start, y_end, leftover_roughness)

    region_w = x_end - x_start
    region_h = y_end - y_start
    region_area = region_w * region_h
    target_area = coverage * region_area

    occupancy = np.zeros((region_h, region_w), dtype=bool)

    margin = 10
    placed_area = 0.0
    max_attempts = 1000

    for _ in range(max_attempts):
        if placed_area >= target_area:
            break

        # Convert the chosen stair_length/stair_width to integers
        rect_w = int(stair_length)
        rect_h = int(stair_width)

        # The maximum placeable top-left x:
        #   region_w - 2*margin is the 'inner' width,
        #   subtract rect_w from that so the rectangle fits.
        avail_w = region_w - 2*margin - rect_w
        avail_h = region_h - 2*margin - rect_h

        # If either dimension is too small, we can't place a ramp
        if avail_w < 0 or avail_h < 0:
            break

        rx = x_start + margin + np.random.randint(0, avail_w + 1)
        ry = y_start + margin + np.random.randint(0, avail_h + 1)

        success = fill_one_stair_rectangle(
            heightmap, occupancy,
            x_start, x_end, y_start, y_end,
            rx, ry, rect_w, rect_h, stair_height
        )
        if success:
            placed_area += rect_w * rect_h


#############################
# Master heightmap generator
#############################
def generate_heightmap(size, n_regions, region_info):
    """
    n_regions in {1,4,9} => 1x1, 2x2, or 3x3 grid.
    Each cell is either "Rough Terrain" or "Staircase."
    """
    heightmap = np.zeros((size, size), dtype=np.float32)
    grid_size = int(np.sqrt(n_regions))
    sub_size = size // grid_size

    for idx in range(n_regions):
        info = region_info[idx]
        row = idx // grid_size
        col = idx % grid_size

        y_start = row * sub_size
        y_end   = (row+1)*sub_size if row < grid_size-1 else size
        x_start = col * sub_size
        x_end   = (col+1)*sub_size if col < grid_size-1 else size

        if info['type'] == 'Rough Terrain':
            fill_with_rough_terrain(heightmap, 
                                    x_start, x_end, y_start, y_end, 
                                    info['roughness'])
        else:
            fill_staircase_region(heightmap,
                                  x_start, x_end,
                                  y_start, y_end,
                                  coverage=info['coverage'],
                                  stair_height=info['stair_height'],
                                  stair_length=info['stair_length'],
                                  stair_width=info['stair_width'],
                                  leftover_roughness=info['leftover_roughness'])
    return heightmap

#############################
# (UI) last_heightmap for saving
#############################
last_heightmap = None

#############################
# Build the UI
#############################
size_slider = widgets.IntSlider(
    value=256, min=256, max=512, step=16,
    description='Size', continuous_update=False
)

n_regions_dropdown = widgets.Dropdown(
    options=[1, 4, 9],
    value=1,
    description='Regions'
)

region_accordions = []
region_type_widgets = []
region_widgets = {
    'roughness': [],
    'coverage': [],
    'stair_height': [],
    'stair_length': [],
    'stair_width': [],
    'leftover_roughness': []
}

for i in range(9):
    rt = widgets.Dropdown(
        options=["Rough Terrain", "Staircase"],
        value="Rough Terrain",
        description="Type",
        style={'description_width': '50px'},
    )
    region_type_widgets.append(rt)

    w_r = widgets.FloatSlider(
        value=0.2, min=0.0, max=1.0, step=0.05,
        description="Rough",
        style={'description_width': '50px'}
    )
    region_widgets['roughness'].append(w_r)

    w_cov = widgets.FloatSlider(
        value=0.5, min=0.0, max=1.0, step=0.05,
        description="Cover",
        style={'description_width': '50px'}
    )
    region_widgets['coverage'].append(w_cov)

    w_h = widgets.FloatSlider(
        value=0.5, min=0.0, max=1.0, step=0.05,
        description="Height",
        style={'description_width': '50px'}
    )
    region_widgets['stair_height'].append(w_h)

    w_len = widgets.FloatSlider(
        value=20.0, min=1.0, max=200.0, step=1.0,
        description="Len",
        style={'description_width': '50px'}
    )
    region_widgets['stair_length'].append(w_len)

    w_wid = widgets.FloatSlider(
        value=20.0, min=1.0, max=200.0, step=1.0,
        description="Width",
        style={'description_width': '50px'}
    )
    region_widgets['stair_width'].append(w_wid)

    w_lor = widgets.FloatSlider(
        value=0.2, min=0.0, max=1.0, step=0.05,
        description="LeftovR",
        style={'description_width': '50px'}
    )
    region_widgets['leftover_roughness'].append(w_lor)

    region_box = widgets.VBox([
        rt, w_r, w_cov, w_h, w_len, w_wid, w_lor
    ])
    acc = widgets.Accordion([region_box])
    acc.set_title(0, f"Region {i+1}")
    acc.selected_index = None
    region_accordions.append(acc)

def update_visibility():
    n = n_regions_dropdown.value
    for i in range(9):
        if i < n:
            region_accordions[i].layout.display = ''
            if region_type_widgets[i].value == "Rough Terrain":
                region_widgets['roughness'][i].disabled = False
                region_widgets['coverage'][i].disabled = True
                region_widgets['stair_height'][i].disabled = True
                region_widgets['stair_length'][i].disabled = True
                region_widgets['stair_width'][i].disabled = True
                region_widgets['leftover_roughness'][i].disabled = True
            else:
                region_widgets['roughness'][i].disabled = True
                region_widgets['coverage'][i].disabled = False
                region_widgets['stair_height'][i].disabled = False
                region_widgets['stair_length'][i].disabled = False
                region_widgets['stair_width'][i].disabled = False
                region_widgets['leftover_roughness'][i].disabled = False
        else:
            region_accordions[i].layout.display = 'none'

def on_n_regions_change(change):
    if change['name'] == 'value':
        update_visibility()

n_regions_dropdown.observe(on_n_regions_change, names='value')

def on_region_type_change(change):
    if change['name'] == 'value':
        update_visibility()

for rt in region_type_widgets:
    rt.observe(on_region_type_change, names='value')

output = widgets.Output()

def generate_and_plot(_=None):
    global last_heightmap
    with output:
        output.clear_output()

        size = size_slider.value
        n_regions = n_regions_dropdown.value

        info_list = []
        for i in range(n_regions):
            rtype = region_type_widgets[i].value
            if rtype == "Rough Terrain":
                rough = region_widgets['roughness'][i].value
                info_list.append({
                    'type': 'Rough Terrain',
                    'roughness': rough
                })
            else:
                cov  = region_widgets['coverage'][i].value
                sth  = region_widgets['stair_height'][i].value
                slen = region_widgets['stair_length'][i].value
                swid = region_widgets['stair_width'][i].value
                lor  = region_widgets['leftover_roughness'][i].value
                info_list.append({
                    'type': 'Staircase',
                    'coverage': cov,
                    'stair_height': sth,
                    'stair_length': slen,
                    'stair_width': swid,
                    'leftover_roughness': lor
                })

        hm = generate_heightmap(size, n_regions, info_list)
        last_heightmap = hm

        plt.figure(figsize=(5,5))
        plt.title(f"Heightmap ({n_regions} region{'s' if n_regions>1 else ''}, size={size})")
        plt.imshow(hm, origin='lower', cmap='gray', vmin=0, vmax=1)
        plt.colorbar(label="Height [0..1]")
        plt.show()

update_button = widgets.Button(description="Generate Heightmap")
update_button.on_click(generate_and_plot)

save_button = widgets.Button(description="Save Heightmap")
def on_save(_):
    global last_heightmap
    if last_heightmap is None:
        print("No heightmap generated yet. Please click Generate first.")
        return

    # Save as PNG (scaled 0..255)
    png_data = (last_heightmap * 255).astype(np.uint8)
    imageio.imwrite("hfields/hfield.png", png_data)
    print("Saved 'hfield.png'.")

save_button.on_click(on_save)

acc_box = widgets.VBox(region_accordions)
top_row = widgets.HBox([size_slider, n_regions_dropdown])
btn_row = widgets.HBox([update_button, save_button])
ui = widgets.VBox([top_row, acc_box, btn_row, output])

update_visibility()
display(ui)


Saved 'heightmap.png'.
Saved 'heightmap.png'.
